In [34]:
import wandb
import pandas as pd
import ydata_profiling


## Load the data from Weights & Biases

Start a W&B run (tagged with the `eda` group) and download the raw `sample.csv` artifact. The file is then read into a pandas DataFrame so we can explore it interactively. 
Using `save_code=True` records this notebook to W&B for reproducibility.

In [35]:
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

## Profile the dataset

We generate an automated profiling report with `ydata_profiling` to get a quick overview of the data: column types, value distributions, missing values, and correlations.

In [ ]:
profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

## Clean the data

Based on the profiling observations, we apply two fixes interactively (these are later formalized in the `basic_cleaning` component):

1. **Drop price outliers** — keep only listings priced between `$10` and `$350`, removing the extreme values that would distort the model.
2. **Convert `last_review`** from string to a proper datetime type.


In [37]:
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

## Verify the cleaned data

We re-run `df.info()` / profiling on the cleaned DataFrame to confirm the
outliers are gone, the price range is sensible, and `last_review` is now a
datetime column.

In [ ]:
df.info()

In [ ]:
df['price'].describe()

In [43]:
run.finish()

## Summary of findings

- The raw data contained **price outliers** and **missing values** that must be
  handled before modeling.
- `last_review` required a **type conversion** to datetime.
- These cleaning steps are now formalized in the `basic_cleaning` component so
  they run automatically as part of the pipeline.
  